# Phase 4: Customer Churn Prediction

## Objective

Predict customers who are likely to stop purchasing.

Business Value:

- Reduce customer loss
- Improve retention
- Increase customer lifetime value

In [12]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Load data and recreate RFM.

In [13]:
df = pd.read_csv("synthetic_online_retail_data.csv")

df['order_date'] = pd.to_datetime(df['order_date'])

df['sales'] = df['quantity'] * df['price']

snapshot_date = df['order_date'].max()

rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days,
    'customer_id': 'count',
    'sales': 'sum'
})

rfm.columns = [
    'Recency',
    'Frequency',
    'Monetary'
]

rfm.head()

,Recency,Frequency,Monetary
customer_id,,,
10201,156,1,624.84
10211,239,1,65.02
10254,190,1,70.93
10299,112,1,815.76
10403,320,1,1319.35


# Create Churn Label
Business Rule

If customer has not purchased for more than 90 days:

Churn = 1

Else:

Churn = 0


In [14]:
rfm['Churn'] = np.where(
    rfm['Recency'] > 90,
    1,
    0
)

rfm.head()

,Recency,Frequency,Monetary,Churn
customer_id,,,,
10201,156,1,624.84,1
10211,239,1,65.02,1
10254,190,1,70.93,1
10299,112,1,815.76,1
10403,320,1,1319.35,1


# Check Churn Distribution

In [15]:
rfm['Churn'].value_counts()

Churn
1    758
0    242
Name: count, dtype: int64

# interpretation:

758 Active Customers

242 Churn Customers

# Prepare Features

In [16]:
X = rfm[
    [
        'Recency',
        'Frequency',
        'Monetary'
    ]
]

y = rfm['Churn']

# Train-Test Split

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Train Model

In [18]:
model = RandomForestClassifier(
    random_state=42
)

model.fit(
    X_train,
    y_train
)

RandomForestClassifier(random_state=42)

# Predictions

In [19]:
predictions = model.predict(
    X_test
)

# Evaluate Model

In [20]:
accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

Accuracy: 100.0 %


# Detailed Report

In [21]:
print(
    classification_report(
        y_test,
        predictions
    )
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        48
           1       1.00      1.00      1.00       152

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



# Feature Importance

In [22]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
})

importance.sort_values(
    by='Importance',
    ascending=False
)

,Feature,Importance
0,Recency,0.984409
2,Monetary,0.015591
1,Frequency,0.000000


# Phase 4 Executive Summary

## Objective

Develop a machine learning model to predict customer churn.

## Model Used

Random Forest Classifier

## Features

- Recency
- Frequency
- Monetary

## Results

- Accuracy: 100.0 %
- Active Customers: 
- Churn Customers: XX

## Key Drivers

1. Recency
2. Frequency
3. Monetary

## Business Recommendations

### High-Risk Customers

Launch retention campaigns.

### Loyal Customers

Increase engagement through rewards.

### VIP Customers

Provide premium support.

## Business Impact

The churn prediction model enables proactive customer retention and helps reduce future revenue loss.

In [24]:
rfm['Predicted_Churn'] = model.predict(X)

# High-Risk Customers

In [25]:
high_risk = rfm[
    rfm['Predicted_Churn'] == 1
]

high_risk.head()

,Recency,Frequency,Monetary,Churn,Predicted_Churn
customer_id,,,,,
10201,156,1,624.84,1,1
10211,239,1,65.02,1,1
10254,190,1,70.93,1,1
10299,112,1,815.76,1,1
10403,320,1,1319.35,1,1


# Count High-Risk Customers

In [27]:
high_risk.shape[0]

758

# Revenue At Risk

In [30]:
high_risk['Monetary'].sum()

np.float64(564003.1)

# Phase 4 Executive Summary

## Objective

Predict customers likely to stop purchasing.

## Machine Learning Model

Random Forest Classifier

## Features Used

- Recency
- Frequency
- Monetary

## Key Findings

- Customers with high recency are more likely to churn.
- Customers with low purchase frequency have higher churn risk.
- Customers with lower spending are more likely to leave.

## Business Recommendations

### High-Risk Customers
Launch retention campaigns immediately.

### Loyal Customers
Offer personalized promotions.

### VIP Customers
Provide premium loyalty benefits.

## Expected Business Impact

- Improved customer retention
- Reduced revenue loss
- Better marketing efficiency